# Population Definition

**Objetivo**

Este notebook tem como objetivo:

- Entender a estrutura das bases disponibilizadas;
- Validar a qualidade e integridade dos dados;
- Analisar os relacionamentos entre as tabelas;
- Definir a população elegível para modelagem;
- Construir os datasets iniciais que serão utilizados nas próximas etapas do projeto.

## 1. Imports e Configurações

In [18]:
import pandas as pd
import numpy as np

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

DATA_PATH = "../data/raw"
OUTPUT_PATH = "../data/processed"

## 2. Carregamento das Bases

### 2.1 Leitura dos Arquivos

In [19]:
base_cadastral = pd.read_parquet(
    f"{DATA_PATH}/base_cadastral.parquet"
)
base_cadastral.head()

,id_cliente,sexo,data_nascimento,qtd_filhos,qtd_membros_familia,renda_anual,tipo_renda,ocupacao,tipo_organizacao,nivel_educacao,estado_civil,tipo_moradia,possui_carro,possui_imovel,nota_regiao_cliente,nota_regiao_cliente_cidade
0,100023,F,1994-01-30,1,2.0,90000.0,State servant,Core staff,Kindergarten,Higher education,Single / not married,House / apartment,N,Y,2,2
1,100031,F,1973-11-13,0,1.0,112500.0,Working,Cooking staff,Business Entity Type 3,Secondary / secondary special,Widow,House / apartment,N,Y,3,2
2,100056,M,1975-02-19,0,2.0,360000.0,Working,Laborers,Transport: type 2,Secondary / secondary special,Married,House / apartment,Y,Y,2,2
3,100069,M,1986-04-10,1,2.0,360000.0,Working,Laborers,Transport: type 4,Secondary / secondary special,Separated,House / apartment,Y,Y,2,2
4,100085,M,1994-07-05,1,3.0,157500.0,Working,Drivers,Business Entity Type 1,Secondary / secondary special,Married,House / apartment,N,Y,2,2


In [20]:
base_submissao = pd.read_parquet(
    f"{DATA_PATH}/base_submissao.parquet"
)
base_submissao.head()

,id_cliente,data_solicitacao,dia_semana_solicitacao,hora_solicitacao,tipo_contrato,valor_credito,valor_bem,valor_parcela
0,100023,2025-02-24,MONDAY,12,Cash loans,544491.0,454500.0,17563.5
1,100031,2025-02-17,MONDAY,9,Cash loans,979992.0,702000.0,27076.5
2,100056,2025-02-20,THURSDAY,10,Cash loans,1506816.0,1350000.0,49927.5
3,100069,2025-02-10,MONDAY,11,Cash loans,640458.0,517500.0,27265.5
4,100085,2025-02-19,WEDNESDAY,12,Cash loans,755190.0,675000.0,28894.5


In [21]:
historico_emprestimos = pd.read_parquet(
    f"{DATA_PATH}/historico_emprestimos.parquet"
)
historico_emprestimos.head()

,id_contrato,id_cliente,tipo_contrato,status_contrato,data_decisao,data_liberacao,data_primeiro_vencimento,data_ultimo_vencimento_original,data_ultimo_vencimento,data_encerramento,valor_solicitado,valor_credito,valor_bem,valor_parcela,valor_entrada,percentual_entrada,qtd_parcelas_planejadas,taxa_juros_padrao,taxa_juros_promocional,tipo_pagamento,finalidade_emprestimo,tipo_cliente,faixa_rendimento,tipo_portfolio,tipo_produto,categoria_bem,combinacao_produto,setor_vendedor,canal_venda,area_venda,dia_semana_solicitacao,hora_solicitacao,flag_ultima_solicitacao_contrato,flag_ultima_solicitacao_dia,motivo_recusa,acompanhantes_cliente,flag_seguro_contratado
0,2802425,108129,Cash loans,Approved,2024-08-29,NaN,2024-09-28,2027-08-14,NaN,NaN,607500.0,679671.0,607500.0,25188.615,NaN,NaN,36.0,NaN,NaN,XNA,XNA,Repeater,low_action,Cash,x-sell,XNA,Cash X-Sell: low,XNA,Contact center,-1,THURSDAY,11,Y,1,XAP,Unaccompanied,1.0
1,2330894,258628,Cash loans,Approved,2022-10-11,NaN,2022-11-10,2024-09-30,2024-08-01,2024-08-04,148500.0,174361.5,148500.0,12165.210,NaN,NaN,24.0,NaN,NaN,Cash through the bank,XNA,Repeater,high,Cash,x-sell,XNA,Cash X-Sell: high,XNA,Credit and cash offices,-1,TUESDAY,15,Y,1,XAP,Unaccompanied,1.0
2,1182516,267782,Cash loans,Approved,2023-04-08,NaN,2023-05-08,2025-09-24,NaN,NaN,405000.0,451777.5,405000.0,20361.600,NaN,NaN,30.0,NaN,NaN,Cash through the bank,XNA,Repeater,low_normal,Cash,x-sell,XNA,Cash X-Sell: low,XNA,Credit and cash offices,-1,SATURDAY,4,Y,1,XAP,NaN,1.0
3,1543131,275707,Cash loans,Approved,2024-02-08,NaN,2024-03-09,2025-02-02,2025-02-02,2025-02-09,229500.0,241920.0,229500.0,22619.520,NaN,NaN,12.0,NaN,NaN,Cash through the bank,XNA,Repeater,low_normal,Cash,x-sell,XNA,Cash X-Sell: low,XNA,Credit and cash offices,-1,THURSDAY,8,Y,1,XAP,Unaccompanied,1.0
4,2261993,299391,Revolving loans,Canceled,2024-09-06,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,XNA,XAP,Repeater,XNA,XNA,XNA,XNA,Card Street,XNA,Credit and cash offices,-1,FRIDAY,13,Y,1,XAP,NaN,NaN


In [22]:
historico_parcelas = pd.read_parquet(
    f"{DATA_PATH}/historico_parcelas.parquet"
)
historico_parcelas.head()

,id_contrato,id_cliente,versao_parcela,numero_parcela,data_prevista_pagamento,data_real_pagamento,valor_previsto_parcela,valor_pago_parcela
0,1594684,100193,0.0,56,2021-12-21,2021-12-21,301.86,301.86
1,1995642,134723,1.0,38,2021-08-09,2021-08-04,12949.20,12949.20
2,1720935,176364,1.0,9,2024-03-06,2024-03-04,61192.53,61192.53
3,1439208,154898,1.0,20,2024-08-11,2024-08-07,8851.23,8851.23
4,1640082,172575,1.0,4,2022-11-15,2022-11-10,8720.28,8720.28


### 2.2 Visão Geral

In [23]:
bases = {
    "base_cadastral": base_cadastral,
    "base_submissao": base_submissao,
    "historico_emprestimos": historico_emprestimos,
    "historico_parcelas": historico_parcelas
}

overview = []

for nome, df in bases.items():

    overview.append({
        "base": nome,
        "linhas": df.shape[0],
        "colunas": df.shape[1]
    })

pd.DataFrame(overview)

,base,linhas,colunas
0,base_cadastral,40000,16
1,base_submissao,40000,8
2,historico_emprestimos,186890,37
3,historico_parcelas,1390978,8


**Principais Observações:**

- A base cadastral contém 40 mil clientes.
- A base de submissão contém 40 mil solicitações para score.
- O histórico possui mais de 186 mil contratos.
- O histórico de parcelas possui aproximadamente 1,4 milhão de registros.

O volume disponível é suficiente para construção de um modelo supervisionado robusto.

## 3. Qualidade dos Dados

### 3.1 Missing Values

In [24]:
for nome, df in bases.items():

    print(f"\n{nome}")

    display(
        pd.DataFrame({
            "missing": df.isna().sum(),
            "missing_pct": (
                df.isna().mean() * 100
            ).round(2)
        })
        .query("missing > 0")
        .sort_values(
            "missing_pct",
            ascending=False
        )
    )


base_cadastral


,missing,missing_pct
ocupacao,12676,31.69



base_submissao


,missing,missing_pct
valor_bem,24,0.06
valor_parcela,4,0.01



historico_emprestimos


,missing,missing_pct
taxa_juros_padrao,186262,99.66
taxa_juros_promocional,186262,99.66
data_liberacao,179790,96.20
data_encerramento,100803,53.94
percentual_entrada,100168,53.60
valor_entrada,100168,53.60
data_ultimo_vencimento,99112,53.03
acompanhantes_cliente,92104,49.28
data_ultimo_vencimento_original,85747,45.88
data_primeiro_vencimento,79745,42.67



historico_parcelas


,missing,missing_pct
data_real_pagamento,339,0.02
valor_pago_parcela,339,0.02


**Principais Observações**

Foram identificadas variáveis com elevado percentual de ausência:

- taxa_juros_padrao
- taxa_juros_promocional
- data_liberacao

Essas variáveis serão reavaliadas nas etapas posteriores para definição de permanência ou exclusão do modelo.

## 4. Integridade Referencial

### 4.1 Clientes sem Cadastro

In [25]:
clientes_sem_cadastro = (
    set(historico_emprestimos["id_cliente"])
    -
    set(base_cadastral["id_cliente"])
)

print(
    f"Clientes sem cadastro: "
    f"{len(clientes_sem_cadastro)}"
)

Clientes sem cadastro: 0


### 4.2 Contratos sem Parcelas

In [26]:
contratos_sem_parcelas = (
    set(historico_emprestimos["id_contrato"])
    -
    set(historico_parcelas["id_contrato"])
)

print(
    f"Contratos sem parcelas: "
    f"{len(contratos_sem_parcelas)}"
)

Contratos sem parcelas: 79471


**Principais Observações**

- Todos os contratos possuem cliente cadastrado.
- Aproximadamente 79 mil contratos não possuem histórico de parcelas.

Esse comportamento sugere a existência de contratos recusados, cancelados ou não efetivados.

## 5. Tratamento de Datas

### 5.1 Conversão

In [27]:
cols_emprestimos = [
    "data_decisao",
    "data_liberacao",
    "data_primeiro_vencimento",
    "data_ultimo_vencimento_original",
    "data_ultimo_vencimento",
    "data_encerramento"
]

for col in cols_emprestimos:
    historico_emprestimos[col] = pd.to_datetime(
        historico_emprestimos[col]
    )

for col in [
    "data_prevista_pagamento",
    "data_real_pagamento"
]:
    historico_parcelas[col] = pd.to_datetime(
        historico_parcelas[col]
    )

base_submissao["data_solicitacao"] = pd.to_datetime(
    base_submissao["data_solicitacao"]
)

base_cadastral["data_nascimento"] = pd.to_datetime(
    base_cadastral["data_nascimento"]
)

## 6. Cobertura Temporal

In [28]:
pd.DataFrame({
    "base": [
        "emprestimos",
        "parcelas"
    ],
    "data_min": [
        historico_emprestimos["data_decisao"].min(),
        historico_parcelas["data_prevista_pagamento"].min()
    ],
    "data_max": [
        historico_emprestimos["data_decisao"].max(),
        historico_parcelas["data_prevista_pagamento"].max()
    ]
})

,base,data_min,data_max
0,emprestimos,2017-02-04,2025-02-22
1,parcelas,2017-02-28,2025-02-22


**Principais Observações**

O histórico disponível cobre aproximadamente oito anos de operação (2017 a 2025), permitindo a construção de variáveis comportamentais e análise longitudinal dos clientes.

## 7. Análise da População

### 7.1 Status dos Contratos

In [29]:
(
    historico_emprestimos["status_contrato"]
    .value_counts()
    .to_frame()
)

,count
status_contrato,
Approved,116182
Canceled,35767
Refused,32108
Unused offer,2833


### 7.2 Contratos por Cliente

In [30]:
(
    historico_emprestimos
    .groupby("id_cliente")
    ["id_contrato"]
    .nunique()
    .describe()
)

count    37952.000000
mean         4.924378
std          4.189973
min          1.000000
25%          2.000000
50%          4.000000
75%          7.000000
max         66.000000
Name: id_contrato, dtype: float64

**Principais Observações**

A maioria dos contratos encontra-se na situação Approved.

Como o objetivo do projeto é modelar inadimplência, apenas contratos efetivamente concedidos serão considerados elegíveis para construção da variável target.

## 8. Definição da População de Modelagem

In [31]:
population_train = (
    historico_emprestimos
    .query(
        "status_contrato == 'Approved'"
    )
    .copy()
)

population_score = (
    base_submissao
    .copy()
)

print(
    f"Treino: {population_train.shape}"
)

print(
    f"Score: {population_score.shape}"
)

Treino: (116182, 37)
Score: (40000, 8)


**Decisão Adotada**

A população de modelagem será composta exclusivamente por contratos com status Approved, uma vez que apenas estes possuem potencial de gerar comportamento de pagamento e inadimplência observável.

Contratos recusados, cancelados ou ofertas não utilizadas serão excluídos do processo de modelagem.

## 9. Persistência

In [32]:
Path(OUTPUT_PATH).mkdir(
    parents=True,
    exist_ok=True
)

population_train.to_parquet(
    f"{OUTPUT_PATH}/population_train.parquet",
    index=False
)

population_score.to_parquet(
    f"{OUTPUT_PATH}/population_score.parquet",
    index=False
)

## 10. Conclusões

Principais achados desta etapa:

- Base histórica com mais de 186 mil contratos.
- Histórico de parcelas com aproximadamente 1,4 milhão de registros.
- Nenhum problema de integridade entre clientes e cadastro.
- Histórico temporal entre 2017 e 2025.
- População elegível definida como contratos Approved.

A próxima etapa consistirá na definição da variável target de inadimplência.